In [ ]:
import pandas as pd
import os
import re
import unicodedata

# ==============================================================================
# CONFIGURACIÓN GENERAL Y NORMALIZACIÓN
# ==============================================================================

LLAVE_PRIMARIA = 'id ine'

COLUMNAS_A_COMPARAR = [
    'empresa', 'direccion', 'colonia', 'cp', 'municipio',
    'entidad federativa', 'version'
]

COLUMNAS_CLAVE = [LLAVE_PRIMARIA] + COLUMNAS_A_COMPARAR


def normalizar_texto(texto):
    """
    Normalización general:
    - Convierte NaN/vacíos en ""
    - Convierte números enteros almacenados como float:
          20210.0 -> 20210
    - Convierte strings numéricos:
          "20210.0" -> "20210"
    - Quita espacios al inicio y final
    - Convierte espacios especiales (NBSP) a espacios normales
    - Reduce múltiples espacios a uno
    - Convierte a minúsculas
    - Elimina acentos
    """

    if pd.isna(texto):
        return ""

    # --------------------------------------------------------------
    # 1. Normalizar tipos numéricos
    # --------------------------------------------------------------

    if isinstance(texto, float):
        if texto.is_integer():
            texto = int(texto)

    # --------------------------------------------------------------
    # 2. Convertir a texto
    # --------------------------------------------------------------

    s = str(texto)

    # Unicode: normaliza caracteres equivalentes
    s = unicodedata.normalize("NFKC", s)

    # Espacios especiales
    s = s.replace("\u00A0", " ")
    s = s.replace("\u2007", " ")
    s = s.replace("\u202F", " ")

    # Quitar espacios extremos
    s = s.strip().lower()

    # --------------------------------------------------------------
    # 3. Corregir números almacenados como texto
    # --------------------------------------------------------------
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]

    # --------------------------------------------------------------
    # 4. Reducir cualquier cantidad de espacios a UNO
    # --------------------------------------------------------------
    s = re.sub(r"\s+", " ", s).strip()

    # --------------------------------------------------------------
    # 5. Eliminar acentos
    # --------------------------------------------------------------
    s = "".join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn"
    )

    return s


def normalizar_empresa(texto):
    """
    Normalización EXCLUSIVA para comparar EMPRESA.

    Corrige diferencias de escritura que no deberían generar
    una discrepancia:

    - Mayúsculas/minúsculas
    - Acentos
    - Espacios adicionales
    - Puntos
    - Comas
    - Signos de puntuación
    - Guiones
    - Paréntesis
    - Abreviaturas comunes
    - Razones sociales escritas con o sin puntos

    Ejemplos:

        S.A. DE C.V.  -> SA DE CV
        S.A DE C.V.   -> SA DE CV
        SA DE CV      -> SA DE CV

        BLVD. FELIPE ANGELES
        BOULEVARD FELIPE ANGELES

        CARR. MEXICO-PACHUCA
        CARRETERA MEXICO PACHUCA

    IMPORTANTE:
    Esto SOLO se utiliza para comparar.
    No modifica los valores originales del reporte.
    """

    s = normalizar_texto(texto)

    if not s:
        return ""

    # --------------------------------------------------------------
    # 1. Normalizar razones sociales
    # --------------------------------------------------------------

    patrones_razon_social = [

        # S.A. DE C.V.
        (
            r'\bs\s*\.?\s*a\s*\.?\s+de\s+c\s*\.?\s*v\s*\.?\b',
            'sa de cv'
        ),

        # S.A DE C.V.
        (
            r'\bsa\s+de\s+cv\b',
            'sa de cv'
        ),

        # S.A.
        (
            r'\bs\s*\.?\s*a\s*\.?\b',
            'sa'
        ),

        # S. DE R.L. DE C.V.
        (
            r'\bs\s*\.?\s+de\s+r\s*\.?\s*l\s*\.?\s+de\s+c\s*\.?\s*v\s*\.?\b',
            's de rl de cv'
        ),

        # S. DE R.L.
        (
            r'\bs\s*\.?\s+de\s+r\s*\.?\s*l\s*\.?\b',
            's de rl'
        ),

        # S. EN C.
        (
            r'\bs\s*\.?\s+en\s+c\s*\.?\b',
            's en c'
        )
    ]

    for patron, reemplazo in patrones_razon_social:

        s = re.sub(
            patron,
            reemplazo,
            s,
            flags=re.IGNORECASE
        )

    # --------------------------------------------------------------
    # 2. Normalizar abreviaturas comunes
    # --------------------------------------------------------------

    abreviaturas = {

        # Vialidades
        r'\bav\.?\b': 'avenida',
        r'\bavda\.?\b': 'avenida',

        r'\bblvd\.?\b': 'boulevard',
        r'\bblvr\.?\b': 'boulevard',

        r'\bcarr\.?\b': 'carretera',
        r'\bctra\.?\b': 'carretera',

        r'\bcalz\.?\b': 'calzada',

        # Colonia / fraccionamiento
        r'\bcol\.?\b': 'colonia',
        r'\bfracc\.?\b': 'fraccionamiento',

        # Número
        r'\bno\.?\b': 'numero',
        r'\bnum\.?\b': 'numero',

        # Orientación
        r'\bpte\.?\b': 'poniente',
        r'\bote\.?\b': 'oriente',
        r'\bnte\.?\b': 'norte',

        # Interior / exterior
        r'\bint\.?\b': 'interior',
        r'\bext\.?\b': 'exterior'
    }

    for patron, reemplazo in abreviaturas.items():

        s = re.sub(
            patron,
            reemplazo,
            s,
            flags=re.IGNORECASE
        )

    # --------------------------------------------------------------
    # 3. Eliminar puntuación que no cambia el significado
    # --------------------------------------------------------------

    s = re.sub(
        r'[.,;:()\[\]{}]',
        '',
        s
    )

    # --------------------------------------------------------------
    # 4. Normalizar diferentes tipos de guiones
    # --------------------------------------------------------------

    s = re.sub(
        r'[-–—]',
        ' ',
        s
    )

    # --------------------------------------------------------------
    # 5. Reducir espacios
    # --------------------------------------------------------------

    s = re.sub(
        r'\s+',
        ' ',
        s
    ).strip()

    # --------------------------------------------------------------
    # 6. Para empresa ignoramos completamente los espacios
    # --------------------------------------------------------------

    s = re.sub(
        r'\s+',
        '',
        s
    )

    return s


def normalizar_para_comparacion(columna, valor):
    """
    Decide qué tipo de normalización utilizar según la columna.
    """

    if columna == 'empresa':
        return normalizar_empresa(valor)

    return normalizar_texto(valor)


def encontrar_e_importar_excel(ruta_archivo):
    """
    Detecta de forma dinámica la fila de inicio buscando
    las columnas clave.
    """

    df_preview = pd.read_excel(
        ruta_archivo,
        header=None,
        nrows=30
    )

    header_row = 0

    for idx, row in df_preview.iterrows():

        row_normalizada = [
            normalizar_texto(celda)
            for celda in row.dropna()
        ]

        coincidencias = sum(
            1
            for col in COLUMNAS_CLAVE
            if col in row_normalizada
        )

        if coincidencias >= 4:
            header_row = idx
            break

    df = pd.read_excel(
        ruta_archivo,
        skiprows=header_row
    )

    df.columns = [
        normalizar_texto(col)
        for col in df.columns
    ]

    # Asegurar que al menos la llave primaria exista
    if LLAVE_PRIMARIA not in df.columns:

        raise ValueError(
            f"No se encontró la columna obligatoria "
            f"'{LLAVE_PRIMARIA.upper()}'"
        )

    return df


# ==============================================================================
# 1. PROCESAR ARCHIVO MADRE
# ==============================================================================

carpeta_madre = 'archivo_madre'

archivos_madre = [
    f
    for f in os.listdir(carpeta_madre)
    if f.endswith(('.xlsx', '.xls'))
]

if not archivos_madre:

    raise FileNotFoundError(
        "❌ No se encontró ningún archivo Excel "
        "en la carpeta 'archivo_madre'."
    )

ruta_madre = os.path.join(
    carpeta_madre,
    archivos_madre[0]
)

print(
    f"📦 Leyendo archivo madre: "
    f"{archivos_madre[0]}..."
)

df_madre_raw = encontrar_e_importar_excel(
    ruta_madre
)


# Limpiar y normalizar el DataFrame Madre
df_madre = pd.DataFrame()

for col in df_madre_raw.columns:

    if col in COLUMNAS_CLAVE:

        df_madre[col] = df_madre_raw[col].apply(
            normalizar_texto
        )


# Eliminar duplicados o filas vacías en la llave
df_madre = df_madre[
    df_madre[LLAVE_PRIMARIA] != ""
].drop_duplicates(
    subset=[LLAVE_PRIMARIA]
)


# Indexar por ID INE
df_madre.set_index(
    LLAVE_PRIMARIA,
    inplace=True
)

print(
    f"✅ Archivo Madre indexado con "
    f"{len(df_madre)} llaves únicas de ID INE.\n"
)


# ==============================================================================
# 2. PROCESAR CARPETA DE INFORMES
# ==============================================================================

carpeta_informes = 'informes'

archivos_informes = [
    f
    for f in os.listdir(carpeta_informes)
    if f.endswith(('.xlsx', '.xls'))
]

if not archivos_informes:

    raise FileNotFoundError(
        "❌ No se encontraron archivos Excel "
        "en la carpeta 'informes'."
    )

print(
    f"📂 Se detectaron "
    f"{len(archivos_informes)} informes para procesar."
)

reporte_diferencias = []

todos_los_ids_informes = set()


for nombre_inf in archivos_informes:

    ruta_inf = os.path.join(
        carpeta_informes,
        nombre_inf
    )

    print(
        f"-> Indexando y cruzando por ID INE: "
        f"{nombre_inf}..."
    )

    try:

        df_inf_raw = encontrar_e_importar_excel(
            ruta_inf
        )

        # ----------------------------------------------------------
        # Limpiar y normalizar el Informe
        # ----------------------------------------------------------

        df_inf = pd.DataFrame()

        for col in df_inf_raw.columns:

            if col in COLUMNAS_CLAVE:

                df_inf[col] = df_inf_raw[col].apply(
                    normalizar_texto
                )

        # ----------------------------------------------------------
        # Procesar fila por fila
        # ----------------------------------------------------------

        for _, fila_inf in df_inf.iterrows():

            id_ine = fila_inf[LLAVE_PRIMARIA]

            if id_ine == "":
                continue

            todos_los_ids_informes.add(
                id_ine
            )

            # ======================================================
            # CASO A:
            # ID INE no existe en Madre
            # ======================================================

            if id_ine not in df_madre.index:

                reporte_diferencias.append({

                    'Archivo de Informe':
                        nombre_inf,

                    'ID INE':
                        id_ine.upper(),

                    'Columna Evaluada':
                        'ID INE',

                    'Tipo de Error':
                        'SOBRA REGISTRO '
                        '(No existe este ID en archivo Madre)',

                    'Valor en Madre':
                        'N/A',

                    'Valor en Informe':
                        id_ine.upper()
                })

                continue

            # ======================================================
            # CASO B:
            # ID existe -> comparar columnas
            # ======================================================

            fila_madre = df_madre.loc[id_ine]

            for col in COLUMNAS_A_COMPARAR:

                if col in fila_inf.index:

                    val_madre = fila_madre[col]

                    val_inf = fila_inf[col]

                    # ------------------------------------------------
                    # NORMALIZACIÓN ESPECÍFICA PARA COMPARACIÓN
                    # ------------------------------------------------

                    comparacion_madre = (
                        normalizar_para_comparacion(
                            col,
                            val_madre
                        )
                    )

                    comparacion_inf = (
                        normalizar_para_comparacion(
                            col,
                            val_inf
                        )
                    )

                    # ------------------------------------------------
                    # Comparar valores ya normalizados
                    # ------------------------------------------------

                    if comparacion_madre != comparacion_inf:

                        reporte_diferencias.append({

                            'Archivo de Informe':
                                nombre_inf,

                            'ID INE':
                                id_ine.upper(),

                            'Columna Evaluada':
                                col.upper(),

                            'Tipo de Error':
                                'DIFERENCIA DE TEXTO '
                                '(Datos no coinciden)',

                            'Valor en Madre':
                                val_madre.upper()
                                if val_madre
                                else "[VACÍO]",

                            'Valor en Informe':
                                val_inf.upper()
                                if val_inf
                                else "[VACÍO]"
                        })

                else:

                    reporte_diferencias.append({

                        'Archivo de Informe':
                            nombre_inf,

                        'ID INE':
                            id_ine.upper(),

                        'Columna Evaluada':
                            col.upper(),

                        'Tipo de Error':
                            'COLUMNA FALTANTE '
                            '(Esta columna no viene en el archivo)',

                        'Valor en Madre':
                            'N/A',

                        'Valor en Informe':
                            'N/A'
                    })

    except Exception as e:

        print(
            f"❌ Error procesando el archivo "
            f"{nombre_inf}: {str(e)}"
        )


# ==============================================================================
# 3. IDENTIFICAR ID INES FALTANTES EN TODOS LOS INFORMES
# ==============================================================================

set_ids_madre = set(
    df_madre.index
)

ids_faltantes_total = (
    set_ids_madre -
    todos_los_ids_informes
)


for id_faltante in ids_faltantes_total:

    reporte_diferencias.append({

        'Archivo de Informe':
            'GLOBAL (Todos los archivos)',

        'ID INE':
            id_faltante.upper(),

        'Columna Evaluada':
            'ID INE',

        'Tipo de Error':
            'FALTA REGISTRO COMPLETO '
            '(Ningún informe incluyó este ID)',

        'Valor en Madre':
            id_faltante.upper(),

        'Valor en Informe':
            'N/A'
    })


# ==============================================================================
# 4. GENERAR EXCEL DE SALIDA
# ==============================================================================

df_resultado = pd.DataFrame(
    reporte_diferencias
)

nombre_salida = (
    "conciliacion_por_id_ine.xlsx"
)


if not df_resultado.empty:

    columnas_reporte = [
        'Archivo de Informe',
        'ID INE',
        'Columna Evaluada',
        'Tipo de Error',
        'Valor en Madre',
        'Valor en Informe'
    ]

    df_resultado = df_resultado[
        columnas_reporte
    ]

    df_resultado.to_excel(
        nombre_salida,
        index=False
    )

    print(
        f"\n¡Proceso finalizado! "
        f"Reporte generado con "
        f"{len(df_resultado)} discrepancias "
        f"basadas en ID INE."
    )

    print(
        f"💾 Descarga el archivo de tus carpetas "
        f"de la izquierda: '{nombre_salida}'"
    )

else:

    print(
        "\n¡Excelente! Todos los IDs coinciden "
        "y sus datos son exactamente iguales "
        "a la Madre."
    )

--- INICIANDO CONCILIACIÓN CON ESCANEO INTELIGENTE DE COLUMNAS ---
✅ Archivo cargado e indexado correctamente: Espectaculares/IP 89.xlsx
✅ Archivo cargado e indexado correctamente: Espectaculares/IP 34.xlsx
✅ Archivo cargado e indexado correctamente: Espectaculares/IP 12.xlsx

📊 Total filas útiles en Testigo: 20
📊 Total filas útiles consolidadas de Espectaculares: 22

🎉 ¡Proceso completado exitosamente con escaneo inteligente!
Tu reporte con las pestañas de errores ya se guardó: 'Conciliacion_Final_Espectaculares.xlsx'


/tmp/ipykernel_1452/2626286011.py:87: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  coincidentes['VERSIÓN_TESTIGO'] = coincidentes['VERSIÓN_TESTIGO'].astype(str).str.strip().str.upper()
/tmp/ipykernel_1452/2626286011.py:88: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  coincidentes['VERSIÓN_OPERATIVO'] = coincidentes['VERSIÓN_OPERATIVO'].astype(str).str.strip().str.upper()
/tmp/ipykernel_1452/2626286011.py:94: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try us

In [ ]:
import pandas as pd
import glob
import os

print("--- INICIANDO AUDITORÍA ESTRICTA (INCLUYE DIRECCIÓN Y CP) ---")

def encontrar_y_cargar_excel(ruta_archivo):
    df_bruto = pd.read_excel(ruta_archivo, header=None, nrows=20)
    fila_titulos = None
    for i, fila in df_bruto.iterrows():
        valores_fila = [str(val).strip().upper() for val in fila.values if pd.notna(val)]
        if 'ID INE' in valores_fila:
            fila_titulos = i
            break
    if fila_titulos is None:
        raise KeyError(f"❌ No se encontró la columna 'ID INE' en: {ruta_archivo}")
    df_limpio = pd.read_excel(ruta_archivo, header=fila_titulos)
    df_limpio.columns = df_limpio.columns.str.strip().str.upper()
    return df_limpio

# 1. CARGAR FUENTES
df_testigo = encontrar_y_cargar_excel("Testigo/TESTIGOS.xlsx")
archivos_operativos = glob.glob("Espectaculares/*.xlsx")
lista_df = [encontrar_y_cargar_excel(f) for f in archivos_operativos]
df_espectaculares = pd.concat(lista_df, ignore_index=True)

col_llave = 'ID INE'
df_testigo[col_llave] = df_testigo[col_llave].astype(str).str.strip().str.upper()
df_espectaculares[col_llave] = df_espectaculares[col_llave].astype(str).str.strip().str.upper()

# Limpiar filas vacías
df_testigo = df_testigo[df_testigo[col_llave].str.contains('INE', na=False)]
df_espectaculares = df_espectaculares[df_espectaculares[col_llave].str.contains('INE', na=False)]

# 2. CRUCE TOTAL (OUTER JOIN)
conciliacion = pd.merge(df_testigo, df_espectaculares, on=col_llave, how='outer', suffixes=('_TESTIGO', '_OPERATIVO'))

# Filas que están en ambos lados para comparar renglón contra renglón
coincidentes = conciliacion.dropna(subset=['NO_TESTIGO', 'NO_OPERATIVO']).copy()

# 3. EVALUACIÓN DE LAS EXCEPCIONES FALTANTES
# A. Error de Código Postal
error_cp = pd.DataFrame()
if 'CP_TESTIGO' in coincidentes.columns and 'CP_OPERATIVO' in coincidentes.columns:
    coincidentes['CP_TESTIGO'] = coincidentes['CP_TESTIGO'].astype(str).str.split('.').str[0].str.strip()
    coincidentes['CP_OPERATIVO'] = coincidentes['CP_OPERATIVO'].astype(str).str.split('.').str[0].str.strip()
    error_cp = coincidentes[coincidentes['CP_TESTIGO'] != coincidentes['CP_OPERATIVO']]

# B. Error de Dirección (Comparación exacta sin limpiar para cachar tu error intencional)
error_direccion = pd.DataFrame()
if 'DIRECCION_TESTIGO' in coincidentes.columns and 'DIRECCION_OPERATIVO' in coincidentes.columns:
    coincidentes['DIRECCION_TESTIGO'] = coincidentes['DIRECCION_TESTIGO'].astype(str).str.strip().str.upper()
    coincidentes['DIRECCION_OPERATIVO'] = coincidentes['DIRECCION_OPERATIVO'].astype(str).str.strip().str.upper()
    error_direccion = coincidentes[coincidentes['DIRECCION_TESTIGO'] != coincidentes['DIRECCION_OPERATIVO']]

# C. Conservar los errores monetarios y de versión anteriores
error_version = coincidentes[coincidentes['VERSIÓN_TESTIGO'].astype(str).str.strip().str.upper() != coincidentes['VERSIÓN_OPERATIVO'].astype(str).str.strip().str.upper()] if 'VERSIÓN_TESTIGO' in coincidentes.columns else pd.DataFrame()
error_costos = coincidentes[pd.to_numeric(coincidentes['TOTAL_TESTIGO'], errors='coerce').round(2) != pd.to_numeric(coincidentes['TOTAL_OPERATIVO'], errors='coerce').round(2)] if 'TOTAL_TESTIGO' in coincidentes.columns else pd.DataFrame()
faltantes_operativos = conciliacion[conciliacion['NO_OPERATIVO'].isna()]

# 4. GUARDAR REPORTE MAESTRO CORREGIDO
archivo_salida = 'Conciliacion_Final_Espectaculares.xlsx'
with pd.ExcelWriter(archivo_salida) as writer:
    if not faltantes_operativos.empty:
        faltantes_operativos[[col_llave]].to_excel(writer, sheet_name='Folios Faltantes', index=False)
    if not error_costos.empty:
        error_costos[[col_llave, 'TOTAL_TESTIGO', 'TOTAL_OPERATIVO']].to_excel(writer, sheet_name='Diferencia Costos', index=False)
    if not error_version.empty:
        error_version[[col_llave, 'VERSIÓN_TESTIGO', 'VERSIÓN_OPERATIVO']].to_excel(writer, sheet_name='Diferencia Versión', index=False)
    if not error_direccion.empty:
        error_direccion[[col_llave, 'DIRECCION_TESTIGO', 'DIRECCION_OPERATIVO']].to_excel(writer, sheet_name='Diferencia Dirección', index=False)
    if not error_cp.empty:
        error_cp[[col_llave, 'CP_TESTIGO', 'CP_OPERATIVO']].to_excel(writer, sheet_name='Diferencia CP', index=False)

print(f"\n🎉 ¡Auditoría completa! Archivo actualizado con éxito.")
print(f"Pestañas creadas: {[reporte for reporte in ['Folios Faltantes', 'Diferencia Costos', 'Diferencia Versión', 'Diferencia Dirección', 'Diferencia CP']]}")


--- INICIANDO AUDITORÍA ESTRICTA (INCLUYE DIRECCIÓN Y CP) ---

🎉 ¡Auditoría completa! Archivo actualizado con éxito.
Pestañas creadas: ['Folios Faltantes', 'Diferencia Costos', 'Diferencia Versión', 'Diferencia Dirección', 'Diferencia CP']
